# Optimización Automática de Hiperparámetros con Optuna

Este cuaderno implementa la sintonía fina de hiperparámetros para el modelo **MTDE-Net** utilizando **Optuna**, una librería de optimización bayesiana de alto rendimiento.

El algoritmo aprende de las ejecuciones previas (a través de TPE - Tree-structured Parzen Estimator) y descarta de manera temprana pruebas poco prometedoras mediante **Poda (Pruning)** para optimizar el tiempo de GPU.

**Nota sobre la reproducibilidad:** En esta versión, se fuerza el reseteo de la semilla global (`set_seed(42)`) al inicio de cada prueba (trial) en la función objetivo. Esto garantiza que todos los modelos comiencen exactamente con los **mismos pesos iniciales**, permitiendo que las variaciones observadas en el rendimiento se deban exclusivamente a los hiperparámetros y no a la suerte de la inicialización aleatoria.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.models.mtde_net import MTDE_Net
from src.loaders.mtde_net_loader import MultimodalThermalDataset
from src.utils import SqrtScaledMSELoss, eval_mtde_net_metrics, split_by_sequence

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

d:\ulima\Ulima_archivos\UL-2026-1\Seminario1\python_tests\flir_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Definición de la Función Objetivo de Optuna (Con Semilla Fija)

In [2]:
def objective(trial):
    # Resetear la semilla al inicio de cada prueba garantiza la misma inicialización de pesos
    set_seed(42)
    
    # 1. Definir el Espacio de Búsqueda para explorar de manera inteligente
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    
    # 2. Cargar Datasets y Dataloaders con el batch size sugerido
    train_ds = MultimodalThermalDataset(metadata_csv="../processed_data/metadata.csv", is_train=True)
    val_ds = MultimodalThermalDataset(metadata_csv="../processed_data/metadata.csv", is_train=False)
    train_ds.root = Path("../processed_data")
    val_ds.root = Path("../processed_data")
    
    t_idx, v_idx = split_by_sequence(train_ds.df)
    
    train_loader = DataLoader(Subset(train_ds, t_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(val_ds, v_idx), batch_size=batch_size)
    
    # 3. Inicializar Modelo con el Dropout dinámico sugerido por Optuna
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = MTDE_Net(dropout=dropout).to(device)
    
    crit = SqrtScaledMSELoss(scale=None)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_mae = float("inf")
    epochs = 30 # Optimizamos rápidamente en 30 épocas por cada prueba
    
    for ep in range(epochs):
        model.train()
        for x_img, x_tab, y in train_loader:
            x_img, x_tab, y = x_img.to(device), x_tab.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x_img, x_tab), y)
            loss.backward()
            opt.step()
            
        # Evaluar métricas en el conjunto de validación en segundos reales
        val_m = eval_mtde_net_metrics(model, val_loader, device, scale=30.0)
        v_mae = val_m["mae"]
        
        # Reportar valor actual a Optuna para toma de decisiones de poda (Pruning)
        trial.report(v_mae, ep)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
        if v_mae < best_mae:
            best_mae = v_mae
            
    return best_mae

### 2. Creación e Inicio del Estudio

In [3]:
# Creamos un estudio que busca minimizar el MAE
study = optuna.create_study(
    direction="minimize", 
    pruner=optuna.pruners.MedianPruner()
)

# Ejecutar la optimización bayesiana
# Puedes configurar n_trials en un número mayor (ej. 30 o 50) para una sintonía profunda
study.optimize(objective, n_trials=30, timeout=3600)

print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS ===")
print(study.best_params)
print(f"Mejor MAE obtenido en validación: {study.best_value:.2f}s")

[I 2026-05-31 16:21:11,886] A new study created in memory with name: no-name-5c1f5591-48ed-4edf-9034-2fea72aceb41
[I 2026-05-31 16:21:57,810] Trial 0 finished with value: 29.75209617614746 and parameters: {'lr': 0.0006107981341921912, 'weight_decay': 0.000907306085933502, 'batch_size': 32, 'dropout': 0.16761892600389622}. Best is trial 0 with value: 29.75209617614746.
[I 2026-05-31 16:22:42,581] Trial 1 finished with value: 24.856414794921875 and parameters: {'lr': 0.00017432922229459466, 'weight_decay': 0.00016393480301478224, 'batch_size': 32, 'dropout': 0.3743019720965546}. Best is trial 1 with value: 24.856414794921875.
[I 2026-05-31 16:23:26,989] Trial 2 finished with value: 25.080596923828125 and parameters: {'lr': 0.0002639824867603867, 'weight_decay': 0.00016783428070315268, 'batch_size': 32, 'dropout': 0.22598980241799344}. Best is trial 1 with value: 24.856414794921875.
[I 2026-05-31 16:24:12,748] Trial 3 finished with value: 23.346118927001953 and parameters: {'lr': 0.000528


=== MEJORES HIPERPARÁMETROS ENCONTRADOS ===
{'lr': 0.0007311180257053372, 'weight_decay': 0.0006371219710483732, 'batch_size': 32, 'dropout': 0.3359529946600127}
Mejor MAE obtenido en validación: 22.32s
